In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,tensorflow as tf,time
from sklearn.metrics import mean_absolute_error,mean_squared_error
from sklearn.preprocessing import MinMaxScaler
np.random.seed(16);tf.random.set_seed(16)
# Two years of daily order counts with weekly and seasonal/festival patterns
dates=pd.date_range("2023-01-01",periods=730,freq="D");t=np.arange(730)
orders=260+35*np.sin(2*np.pi*t/7)+22*np.sin(2*np.pi*t/365)+np.where((t%120)<5,55,0)+np.random.normal(0,10,730)
series=pd.Series(orders,index=dates);sc=MinMaxScaler();z=sc.fit_transform(series.values.reshape(-1,1)).ravel()
lookback=30;X=np.array([z[i-lookback:i] for i in range(lookback,len(z))])[...,None];y=z[lookback:]
split=int(.8*len(X));Xtr,Xte,ytr,yte=X[:split],X[split:],y[:split],y[split:]
actual=sc.inverse_transform(yte.reshape(-1,1)).ravel();rows=[];predictions={};histories={}
for name,layer in [("SimpleRNN",tf.keras.layers.SimpleRNN),("LSTM",tf.keras.layers.LSTM)]:
 m=tf.keras.Sequential([tf.keras.layers.Input((lookback,1)),layer(32),tf.keras.layers.Dense(1)])
 m.compile(optimizer="adam",loss="mse");start=time.time();h=m.fit(Xtr,ytr,validation_split=.1,epochs=20,batch_size=32,verbose=1);elapsed=time.time()-start
 p=sc.inverse_transform(m.predict(Xte,verbose=0)).ravel();predictions[name]=p;histories[name]=h.history
 mae=mean_absolute_error(actual,p);rmse=np.sqrt(mean_squared_error(actual,p));mape=np.mean(np.abs((actual-p)/np.maximum(np.abs(actual),1e-8)))*100
 rows.append([name,mae,rmse,mape,elapsed,m.count_params()])
df=pd.DataFrame(rows,columns=["Model","MAE","RMSE","MAPE (%)","Training seconds","Parameters"])
print(df.round(3).to_string(index=False))
plt.figure(figsize=(12,4));plt.plot(actual[:120],label="Actual")
for name,p in predictions.items():plt.plot(p[:120],label=name)
plt.title("Daily order forecast: actual vs SimpleRNN vs LSTM");plt.xlabel("Test day");plt.ylabel("Orders");plt.legend();plt.grid();plt.show()
fig,ax=plt.subplots(1,2,figsize=(11,4))
for name,h in histories.items():ax[0].plot(h["loss"],label=name);ax[1].plot(h["val_loss"],label=name)
ax[0].set_title("Training loss");ax[1].set_title("Validation loss")
for a in ax:a.set_xlabel("Epoch");a.set_ylabel("MSE");a.legend();a.grid()
plt.tight_layout();plt.show()